---
**Pipeline:** [01 Dataset] → [02 Architecture] → [03 Optimization] → **[04 Training]** → [05 Evaluation] → [06 Export]

**Prev:** `03_training_optimization.ipynb` | **Next:** `05_model_evaluation.ipynb`

---

# 🔥 04 - Model Training

## Train MobileFaceNet + ArcFace on VGGFace2

This is the main training notebook with Colab disconnect protection.

**Features:**
- 🚀 Mixed precision training
- 💾 Auto-checkpoint every 500 steps
- 🔄 Auto-resume on Colab restart
- 📊 Real-time metrics logging

**Estimated Time:** 2-4 hours on Colab T4/V100

---

## 1. Environment Setup

In [ ]:
# Setup optimized paths for Colab
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
    print("🌐 Running in Google Colab")
    # Data in /content (fast), Models in Drive (persistent)
    PROJECT_ROOT = Path("/content/face_recognition")
    drive.mount('/content/drive')
    DRIVE_ROOT = Path("/content/drive/MyDrive/face_based_attendance_system")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    print("⚡ OPTIMIZATION: Data in /content (fast ephemeral)")
    print("💾 SAFETY: Checkpoints in Drive (persistent)")
    print("💾 SAFETY: Processed data backed up in Drive")
    print("⚠️  Tip: Data will be synced from Drive to local storage")
except ImportError:
    IN_COLAB = False
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DRIVE_ROOT = None
    print("💻 Running locally")

sys.path.insert(0, str(PROJECT_ROOT))

# Set paths
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"

# Checkpoints go to Drive in Colab, local otherwise
if IN_COLAB and DRIVE_ROOT:
    CHECKPOINT_DIR = DRIVE_ROOT / "models" / "checkpoints"
else:
    CHECKPOINT_DIR = PROJECT_ROOT / "models" / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Directory Structure:")
print(f"   Data: {DATA_DIR}")
print(f"   Processed (local): {PROCESSED_DIR}")
print(f"   Training data: {TRAIN_DIR}")
print(f"   Validation data: {VAL_DIR}")
print(f"   Checkpoints: {CHECKPOINT_DIR}")
if DRIVE_ROOT:
    print(f"   Drive backup: {DRIVE_ROOT / 'data' / 'processed'}")

🌐 Running in Google Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚡ OPTIMIZATION: Data in /content (fast ephemeral)
💾 SAFETY: Checkpoints in Drive (persistent)
💾 SAFETY: Processed data backed up in Drive
⚠️  Tip: Data will be synced from Drive to local storage

📂 Directory Structure:
   Data: /content/face_recognition/data
   Processed (local): /content/face_recognition/data/processed
   Checkpoints: /content/drive/MyDrive/face_based_attendance_system/models/checkpoints
   Drive backup: /content/drive/MyDrive/face_based_attendance_system/data/processed


In [37]:
# Install requirements
!pip install -q albumentations tqdm tensorboard scikit-learn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast
from torch.optim import SGD
from torch.optim.lr_scheduler import CosineAnnealingLR

import numpy as np
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm
import math
import time
from datetime import datetime
from dataclasses import dataclass
from typing import Optional, Dict
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🔧 Device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


🔧 Device: cuda
   GPU: Tesla T4
   Memory: 14.6 GB


## 2. Configuration

In [ ]:
@dataclass
class TrainingConfig:
    """Training configuration optimized for Colab."""
    # Paths - use the train/val directories created by the dataset split
    train_dir: Path = TRAIN_DIR  # DATA_DIR / "train"
    val_dir: Path = VAL_DIR      # DATA_DIR / "val"
    checkpoint_dir: Path = CHECKPOINT_DIR

    # Model
    embedding_dim: int = 512

    # ArcFace
    arcface_scale: float = 64.0
    arcface_margin: float = 0.5

    # ✅ Margin warmup: prevents 0% accuracy at start of training
    # Starts with no margin (softmax-like), gradually increases to full margin
    margin_warmup_epochs: int = 5
    arcface_margin_start: float = 0.0
    arcface_margin_end: float = 0.5

    # Training
    batch_size: int = 256 if torch.cuda.is_available() else 32 # Optimized batch_size
    num_epochs: int = 25
    # ✅ Lowered from 0.1 → 0.01 for stable ArcFace training from scratch
    # 0.1 is standard for large-batch distributed training; 0.01 is better for single-GPU
    base_lr: float = 0.01
    weight_decay: float = 5e-4
    momentum: float = 0.9

    # Schedule
    warmup_epochs: float = 1.0
    min_lr: float = 1e-6

    # Mixed Precision
    use_amp: bool = torch.cuda.is_available()
    grad_clip: float = 5.0

    # Checkpointing (Colab protection!)
    checkpoint_every_n_steps: int = 500
    keep_last_n_checkpoints: int = 3

    # Data - GPU optimization settings
    num_workers: int = 6 if IN_COLAB else 8 # Optimized num_workers
    pin_memory: bool = True
    prefetch_factor: int = 3  # Pre-load batches for GPU utilization
    persistent_workers: bool = True  # Keep workers alive between epochs
    non_blocking: bool = True  # Async GPU transfers

config = TrainingConfig()
print(f"✅ Training config created")
print(f"   Train: {config.train_dir}")
print(f"   Val: {config.val_dir}")
print(f"   Checkpoints: {config.checkpoint_dir}")
print(f"\n🚀 GPU Optimization Settings:")
print(f"   batch_size: {config.batch_size}")
print(f"   num_workers: {config.num_workers} (parallel data loading)")
print(f"   prefetch_factor: {config.prefetch_factor} ({config.num_workers * config.prefetch_factor} batches pre-loaded)")
print(f"   persistent_workers: {config.persistent_workers}")
print(f"   non_blocking: {config.non_blocking}")

## 3. Data Loading

In [39]:
# Augmentations
# NOTE: albumentations v2.x replaced GaussNoise(var_limit=...) with GaussNoise(std_range=...)
# std_range is in [0,1] scale. Old var_limit=(5,30) on [0,255] ≈ std_range=(0.01,0.02)
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(std_range=(0.01, 0.05), p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    A.Rotate(limit=10, border_mode=0, p=0.3),
    A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ToTensorV2(),
])

print("✅ Transforms defined")

✅ Transforms defined


In [40]:
from torch.utils.data import Dataset, Subset
from sklearn.model_selection import train_test_split

class FaceDataset(Dataset):
    """Face dataset with identity labels."""

    def __init__(self, root_dir: Path, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        self.class_to_idx = {}

        # Build index
        if self.root_dir.exists():
            for idx, class_dir in enumerate(sorted(self.root_dir.iterdir())):
                if class_dir.is_dir():
                    self.class_to_idx[class_dir.name] = idx
                    for img_path in class_dir.glob('*.jpg'):
                        self.samples.append((img_path, idx))
                    for img_path in class_dir.glob('*.png'):
                        self.samples.append((img_path, idx))

        self.num_classes = len(self.class_to_idx)
        print(f"📊 Dataset: {len(self.samples):,} images, {self.num_classes:,} classes")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = np.array(Image.open(img_path).convert('RGB'))

        if self.transform:
            img = self.transform(image=img)['image']

        return img, label

# ============================================================================
# CREATE DATASETS WITH CLASS-COMPATIBILITY CHECK
# ============================================================================
# ✅ FIX: ArcFace classification-based validation REQUIRES matching class indices.
#    VGGFace2 train/test sets have DIFFERENT identities → val labels don't map
#    to the ArcFace head's weight matrix → 0% val accuracy guaranteed.
#    Solution: Always auto-split from training data when classes differ.

print("\nLoading datasets...")
train_dataset_full = FaceDataset(config.train_dir, train_transform)
val_dataset_raw = FaceDataset(config.val_dir, val_transform)

# Determine if we need to auto-split from training data
need_auto_split = False

if len(val_dataset_raw) == 0:
    need_auto_split = True
    print("\n⚠️  No validation data found → will auto-split from training data")
elif len(train_dataset_full) > 0:
    # Check if val classes match train classes
    # ArcFace head has weights for TRAINING classes only.
    # If val has DIFFERENT OR FEWER classes (even a subset!), the
    # class-to-index mapping won't match → 0% accuracy guaranteed.
    train_classes = set(train_dataset_full.class_to_idx.keys())
    val_classes = set(val_dataset_raw.class_to_idx.keys())
    overlap = train_classes & val_classes

    if val_classes != train_classes:
        need_auto_split = True
        print(f"\n⚠️  Validation identities DIFFER from training!")
        print(f"   Train: {len(train_classes):,} classes, Val: {len(val_classes):,} classes")
        print(f"   Overlap: {len(overlap):,} classes ({100*len(overlap)/max(1,len(val_classes)):.0f}%)")
        print(f"   → ArcFace classification requires matching classes")
        print(f"   → Auto-splitting from training data for accurate metrics")

if need_auto_split and len(train_dataset_full) > 0:
    print("\n🔄 Creating stratified train/val split (90/10) from training data...")

    # Get labels for stratified split
    all_labels = [label for _, label in train_dataset_full.samples]

    # ✅ Handle classes with only 1 sample (stratified split requires >= 2)
    from collections import Counter
    label_counts = Counter(all_labels)
    single_sample_classes = {label for label, count in label_counts.items() if count < 2}

    if single_sample_classes:
        print(f"   ⚠️  Found {len(single_sample_classes):,} classes with only 1 sample")
        print(f"   → Assigning them to training set (stratify requires >= 2 samples/class)")

        # Separate single-sample indices (go to train) from multi-sample indices (stratified split)
        single_sample_indices = [i for i, label in enumerate(all_labels) if label in single_sample_classes]
        multi_sample_indices = [i for i, label in enumerate(all_labels) if label not in single_sample_classes]
        multi_sample_labels = [all_labels[i] for i in multi_sample_indices]

        # Stratified split on multi-sample classes only
        train_multi, val_multi = train_test_split(
            multi_sample_indices,
            test_size=0.1,
            stratify=multi_sample_labels,
            random_state=42
        )

        # Add single-sample classes to training set
        train_indices = train_multi + single_sample_indices
        val_indices = val_multi
    else:
        # All classes have >= 2 samples, standard stratified split
        train_indices, val_indices = train_test_split(
            range(len(train_dataset_full)),
            test_size=0.1,
            stratify=all_labels,
            random_state=42
        )

    print(f"   Original: {len(train_dataset_full):,} images")
    print(f"   → Train: {len(train_indices):,} ({len(train_indices)/len(train_dataset_full)*100:.1f}%)")
    print(f"   → Val:   {len(val_indices):,} ({len(val_indices)/len(train_dataset_full)*100:.1f}%)")

    # Create subsets with appropriate transforms
    train_dataset = Subset(train_dataset_full, train_indices)

    # For validation, create a new dataset with val_transform (no augmentation)
    val_dataset_full = FaceDataset(config.train_dir, val_transform)
    val_dataset = Subset(val_dataset_full, val_indices)

    NUM_CLASSES = train_dataset_full.num_classes
    print(f"\n✅ Split created! Classes: {NUM_CLASSES:,}")
else:
    train_dataset = train_dataset_full
    val_dataset = val_dataset_raw
    NUM_CLASSES = train_dataset.num_classes
    if len(val_dataset) > 0:
        print(f"\n✅ Using existing validation data: {len(val_dataset):,} images (classes match training)")
    print(f"✅ Number of classes: {NUM_CLASSES:,}")


Loading datasets...


AttributeError: 'TrainingConfig' object has no attribute 'train_dir'

In [ ]:
# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=config.pin_memory,
    drop_last=True,
    persistent_workers=config.num_workers > 0,
    prefetch_factor=config.prefetch_factor if config.num_workers > 0 else None,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=config.pin_memory,
    persistent_workers=config.num_workers > 0,
    prefetch_factor=config.prefetch_factor if config.num_workers > 0 else None,
)

print(f"✅ Train batches: {len(train_loader):,}")
print(f"✅ Val batches: {len(val_loader):,}")

## 4. Model Definition

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, p=1, g=1):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, k, s, p, groups=g, bias=False)
        self.bn = nn.BatchNorm2d(out_c)
        self.act = nn.PReLU(out_c)
    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class DepthWise(nn.Module):
    def __init__(self, in_c, out_c, s=1):
        super().__init__()
        self.dw = nn.Conv2d(in_c, in_c, 3, s, 1, groups=in_c, bias=False)
        self.bn1 = nn.BatchNorm2d(in_c)
        self.act1 = nn.PReLU(in_c)
        self.pw = nn.Conv2d(in_c, out_c, 1, 1, 0, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.act2 = nn.PReLU(out_c)
    def forward(self, x):
        x = self.act1(self.bn1(self.dw(x)))
        return self.act2(self.bn2(self.pw(x)))

class DepthWiseRes(nn.Module):
    def __init__(self, in_c, out_c, s=1):
        super().__init__()
        self.residual = (s == 1 and in_c == out_c)
        self.dw = DepthWise(in_c, out_c, s)
    def forward(self, x):
        return x + self.dw(x) if self.residual else self.dw(x)

def make_stage(in_c, out_c, n, s=2):
    layers = [DepthWiseRes(in_c, out_c, s)]
    for _ in range(1, n):
        layers.append(DepthWiseRes(out_c, out_c, 1))
    return nn.Sequential(*layers)

class MobileFaceNet(nn.Module):
    def __init__(self, emb_dim=512):
        super().__init__()
        self.conv1 = ConvBlock(3, 64, 3, 2, 1)
        self.dw1 = DepthWise(64, 64, 1)
        self.stage1 = make_stage(64, 64, 5, 2)
        self.stage2 = make_stage(64, 128, 1, 2)
        self.stage3 = make_stage(128, 128, 6, 2)
        self.stage4 = make_stage(128, 128, 1, 1)
        self.conv_exp = ConvBlock(128, 512, 1, 1, 0)
        self.gdc = nn.Sequential(
            nn.Conv2d(512, 512, 7, 1, 0, groups=512, bias=False),
            nn.BatchNorm2d(512),
        )
        self.fc = nn.Linear(512, emb_dim, bias=False)
        self.bn = nn.BatchNorm1d(emb_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')

    def forward(self, x):
        x = self.conv1(x)
        x = self.dw1(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.conv_exp(x)
        x = self.gdc(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        x = self.bn(x)
        return F.normalize(x, p=2, dim=1)

print("✅ MobileFaceNet defined")

In [ ]:
class ArcFace(nn.Module):
    """
    ArcFace with margin warmup support and numerically stable implementation.

    Key improvements over naive implementation:
    - Uses arccos-based margin (matches official insightface) for FP16 stability
    - Stores raw cosine logits for accurate training accuracy measurement
    - Supports dynamic margin via set_margin() for margin warmup
    """
    def __init__(self, in_features, out_features, s=64.0, m=0.5):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self._last_cos = None  # Store raw cosine for accuracy computation

    def set_margin(self, m):
        """Dynamically update margin (for margin warmup)."""
        self.m = m

    def forward(self, emb, labels):
        # ✅ Force FP32 for numerical stability under AMP
        emb_fp32 = emb.float()
        w = F.normalize(self.weight.float(), p=2, dim=1)
        cos = F.linear(emb_fp32, w).clamp(-1 + 1e-7, 1 - 1e-7)

        # Store raw cosine for accuracy measurement (before margin)
        self._last_cos = cos.detach()

        # ✅ Numerically stable: arccos → add margin → cos (matches insightface official)
        # This avoids sqrt(1 - cos²) which is unstable in FP16
        theta = cos.arccos()
        target_logit = cos[torch.arange(cos.size(0), device=cos.device), labels]
        target_theta = target_logit.arccos()
        target_theta_m = target_theta + self.m

        # Clamp to valid range [0, pi]
        target_theta_m = target_theta_m.clamp(0, math.pi)
        cos_theta_m = target_theta_m.cos()

        # Replace correct-class cosine with margin-penalized version
        one_hot = F.one_hot(labels, self.weight.size(0)).float()
        logits = cos * (1 - one_hot) + cos_theta_m.unsqueeze(1) * one_hot

        # Scale
        logits = logits * self.s
        return logits

class MobileArcFace(nn.Module):
    def __init__(self, num_classes, emb_dim=512, s=64.0, m=0.5):
        super().__init__()
        self.backbone = MobileFaceNet(emb_dim)
        self.head = ArcFace(emb_dim, num_classes, s, m)

    def forward(self, x, labels=None):
        emb = self.backbone(x)
        if labels is not None:
            return self.head(emb, labels)
        return emb

print("✅ ArcFace and MobileArcFace defined")
print("   Features: margin warmup support, FP32 stability, raw cosine tracking")

In [ ]:
# ============================================================================
# 🚀 GPU OPTIMIZATION: Enable cuDNN Benchmark & Channels-Last Memory Format
# ============================================================================
# These settings can provide 10-30% speedup for convolutional models
import torch

# Enable cuDNN benchmark for optimal CUDA kernel selection
torch.backends.cudnn.benchmark = True

# Enable TF32 for Ampere+ GPUs (3x matmul speedup with minimal precision loss)
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("✅ TF32 enabled (Ampere+ GPU detected)")

print("✅ cuDNN benchmark mode enabled (optimal CUDA kernels)")

# Create model
model = MobileArcFace(
    num_classes=max(NUM_CLASSES, 2),  # At least 2 classes
    emb_dim=config.embedding_dim,
    s=config.arcface_scale,
    m=config.arcface_margin,
).to(device)

# Apply channels-last memory format for 10-20% speedup in conv layers
model = model.to(memory_format=torch.channels_last)
print("✅ Channels-last memory format enabled (10-20% conv speedup)")

# Count parameters
backbone_params = sum(p.numel() for p in model.backbone.parameters())
head_params = sum(p.numel() for p in model.head.parameters())
total_params = backbone_params + head_params

print(f"\n📊 Model Statistics:")
print(f"   Backbone: {backbone_params:,} params")
print(f"   Head: {head_params:,} params")
print(f"   Total: {total_params:,} params")
print(f"   Backbone size: ~{backbone_params * 4 / 1024**2:.1f} MB")

## 5. Training Setup - Complete Initialization

In [ ]:
# ============================================================================
# 🔥 COMPLETE TRAINING SETUP AND INITIALIZATION
# ============================================================================
# Run this cell FIRST before training!

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast
from torch.optim import SGD
from tqdm.auto import tqdm
import numpy as np
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import math
import time
from datetime import datetime
from dataclasses import dataclass
from pathlib import Path

print("="*70)
print("🔥 COMPLETE TRAINING SETUP - INITIALIZING")
print("="*70)

# ============================================================================
# 1. DEVICE & ENVIRONMENT
# ============================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✅ Device: {device}")

# ============================================================================
# 2. PATHS SETUP
# ============================================================================
try:
    from google.colab import drive
    IN_COLAB = True
    PROJECT_ROOT = Path("/content/face_recognition")

    # Mount Google Drive if not mounted
    if not Path("/content/drive").exists() or not Path("/content/drive/MyDrive").exists():
        drive.mount('/content/drive', force_remount=False)

    DRIVE_ROOT = Path("/content/drive/MyDrive/face_based_attendance_system")
    CHECKPOINT_DIR = DRIVE_ROOT / "models" / "checkpoints"
except ImportError:
    IN_COLAB = False
    current_path = Path.cwd()
    PROJECT_ROOT = current_path
    while PROJECT_ROOT.name not in ['face-based_attendance_system', ''] and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent
    if PROJECT_ROOT.name == '':
        PROJECT_ROOT = Path.cwd()
    CHECKPOINT_DIR = PROJECT_ROOT / "models" / "checkpoints"

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Checkpoint dir: {CHECKPOINT_DIR}")
print(f"✅ Data dir: {DATA_DIR}")

# ============================================================================
# 3. TRAINING CONFIGURATION
# ============================================================================
@dataclass
class TrainingConfig:
    """Training configuration optimized for Colab."""
    # Paths - use TRAIN_DIR/VAL_DIR from Section 1 (dataset download)
    train_dir: Path = TRAIN_DIR
    val_dir: Path = VAL_DIR
    checkpoint_dir: Path = CHECKPOINT_DIR

    # Model
    embedding_dim: int = 512

    # ArcFace
    arcface_scale: float = 64.0
    arcface_margin: float = 0.5

    # ✅ Margin warmup: prevents 0% accuracy at start of training
    margin_warmup_epochs: int = 5
    arcface_margin_start: float = 0.0
    arcface_margin_end: float = 0.5

    # Training
    batch_size: int = 128 if torch.cuda.is_available() else 32
    num_epochs: int = 25
    # ✅ Lowered from 0.1 → 0.01 for stable ArcFace training from scratch
    base_lr: float = 0.01
    weight_decay: float = 5e-4
    momentum: float = 0.9

    # Schedule
    warmup_epochs: float = 1.0
    min_lr: float = 1e-6

    # Mixed Precision
    use_amp: bool = torch.cuda.is_available()
    grad_clip: float = 5.0

    # Checkpointing
    checkpoint_every_n_steps: int = 500
    keep_last_n_checkpoints: int = 3

    # Data
    num_workers: int = 2 if IN_COLAB else 4
    pin_memory: bool = True

config = TrainingConfig()

print("\n📋 Training Config:")
print(f"   Batch size: {config.batch_size}")
print(f"   Epochs: {config.num_epochs}")
print(f"   Base LR: {config.base_lr}")
print(f"   AMP: {config.use_amp}")
print(f"   ArcFace margin: {config.arcface_margin_start} → {config.arcface_margin_end} (warmup: {config.margin_warmup_epochs} epochs)")
print(f"   Checkpoint every: {config.checkpoint_every_n_steps} steps")

# ============================================================================
# 4. OPTIMIZER & SCHEDULER
# ============================================================================
# Note: Make sure 'model' variable exists before running this
if 'model' not in globals():
    print("\n⚠️ WARNING: 'model' not found!")
    print("   Run the model creation cell first, then run this cell again.")
else:
    optimizer = SGD(
        model.parameters(),
        lr=config.base_lr,
        momentum=config.momentum,
        weight_decay=config.weight_decay,
    )

    # Calculate training schedule
    if 'train_loader' not in globals():
        print("\n⚠️ WARNING: 'train_loader' not found!")
        print("   Run the data loading cell first, then run this cell again.")
    else:
        total_steps = len(train_loader) * config.num_epochs
        warmup_steps = int(len(train_loader) * config.warmup_epochs)

        # AMP scaler
        scaler = GradScaler('cuda', enabled=config.use_amp)

        print(f"\n✅ Optimizer: SGD (lr={config.base_lr}, momentum={config.momentum})")
        print(f"✅ Total steps: {total_steps:,}")
        print(f"✅ Warmup steps: {warmup_steps:,}")
        print(f"✅ AMP Scaler: Enabled={config.use_amp}")

# ============================================================================
# 5. HELPER FUNCTIONS
# ============================================================================

def get_lr(step, warmup_steps, total_steps, base_lr, min_lr=1e-6):
    """Warmup + Cosine decay learning rate schedule."""
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))

def set_lr(optimizer, lr):
    """Set learning rate for all parameter groups."""
    for g in optimizer.param_groups:
        g['lr'] = lr

print("\n✅ Helper functions defined")

print("\n" + "="*70)
print("✅ SETUP COMPLETE! Ready for training functions.")
print("="*70)

## 6. Checkpoint Management Functions

In [ ]:
def save_checkpoint(path, model, optimizer, scaler, epoch, step, loss, verbose=True):
    """Save training checkpoint."""
    checkpoint = {
        'epoch': epoch,
        'step': step,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scaler': scaler.state_dict(),
        'loss': loss,
        'timestamp': datetime.now().isoformat(),
    }
    torch.save(checkpoint, path)
    if verbose:
        print(f"💾 Checkpoint saved: {path.name}")

def load_checkpoint(path, model, optimizer=None, scaler=None):
    """Load training checkpoint."""
    print(f"📂 Loading checkpoint: {path}")
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model'])
    if optimizer and 'optimizer' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer'])
    if scaler and 'scaler' in ckpt:
        scaler.load_state_dict(ckpt['scaler'])
    print(f"✅ Resumed from epoch {ckpt['epoch']}, step {ckpt['step']}")
    return ckpt['epoch'], ckpt['step']

def find_latest_checkpoint(ckpt_dir):
    """Find most recent checkpoint."""
    latest = ckpt_dir / 'checkpoint_latest.pth'
    if latest.exists():
        return latest
    # Fall back to numbered checkpoints
    ckpts = list(ckpt_dir.glob('checkpoint_epoch*.pth'))
    if ckpts:
        return max(ckpts, key=lambda p: p.stat().st_mtime)
    return None

print("✅ Checkpoint functions defined")

## 7. Training and Validation Functions

In [ ]:
def train_epoch(model, loader, optimizer, scaler, epoch, global_step, config, warmup_steps, total_steps):
    """Train for one epoch."""
    model.train()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    pbar = tqdm(loader,
                desc=f"Epoch {epoch+1}/{config.num_epochs}",
                ncols=120,
                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}] {postfix}')

    for batch_idx, (images, labels) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Update LR
        lr = get_lr(global_step, warmup_steps, total_steps, config.base_lr, config.min_lr)
        set_lr(optimizer, lr)

        # Forward
        optimizer.zero_grad()
        with autocast(device_type='cuda', enabled=config.use_amp):
            logits = model(images, labels)
            loss = F.cross_entropy(logits, labels)

        # Backward
        scaler.scale(loss).backward()

        # Gradient clipping
        if config.use_amp:
            scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)

        scaler.step(optimizer)
        scaler.update()

        # ✅ FIX: Compute accuracy on RAW cosine logits (before margin penalty)
        # ArcFace margin makes correct-class logit lower → argmax never picks correct class
        # Using raw cosine gives true measure of model's discriminative ability
        with torch.no_grad():
            if hasattr(model, 'head') and model.head._last_cos is not None:
                pred = model.head._last_cos.argmax(dim=1)
            else:
                pred = logits.argmax(dim=1)
            correct = (pred == labels).sum().item()

        total_loss += loss.item() * images.size(0)
        total_correct += correct
        total_samples += images.size(0)

        global_step += 1

        # Update progress bar
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100*total_correct/total_samples:.1f}%',
            'lr': f'{lr:.6f}',
        })

        # Checkpoint (silent for frequent saves)
        if global_step % config.checkpoint_every_n_steps == 0:
            save_checkpoint(
                CHECKPOINT_DIR / 'checkpoint_latest.pth',
                model, optimizer, scaler, epoch, global_step, loss.item(),
                verbose=False
            )

    avg_loss = total_loss / total_samples
    avg_acc = 100 * total_correct / total_samples

    return avg_loss, avg_acc, global_step

print("✅ Training function defined (with raw cosine accuracy)")

In [ ]:
@torch.no_grad()
def validate(model, loader, config):
    """Validate using raw cosine logits (no ArcFace margin penalty).

    ArcFace adds angular margin to the ground-truth class during forward pass,
    which makes the loss HIGHER (harder). For monitoring purposes, we compute
    val loss on RAW cosine similarities × scale factor, giving a true picture
    of how well the model separates identities.
    """
    model.eval()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast(device_type='cuda', enabled=config.use_amp):
            # Forward pass — this populates model.head._last_cos
            _ = model(images, labels)

            # ✅ Use RAW cosine similarity (no margin) for val loss
            # This gives a true measure of model quality
            raw_cos = model.head._last_cos
            raw_logits = raw_cos * model.head.s  # scale factor only, no margin
            loss = F.cross_entropy(raw_logits, labels)

        # Accuracy on raw cosine (before margin)
        pred = raw_cos.argmax(dim=1)
        correct = (pred == labels).sum().item()

        total_loss += loss.item() * images.size(0)
        total_correct += correct
        total_samples += images.size(0)

    avg_loss = total_loss / max(1, total_samples)
    avg_acc = 100 * total_correct / max(1, total_samples)

    return avg_loss, avg_acc

print("✅ Validation function defined (raw cosine logits, no margin penalty)")

In [ ]:
@torch.no_grad()
def validate_embeddings(model, loader, config, num_pairs=3000):
    """
    Embedding-based face verification validation (industry standard).

    Instead of classification accuracy (which requires matching class indices),
    this measures the model's actual face recognition quality by:
    1. Extracting embeddings from the backbone (no ArcFace head)
    2. Generating positive pairs (same identity) and negative pairs (different identity)
    3. Computing cosine similarity and finding optimal threshold
    4. Reporting verification accuracy, TAR@FAR=0.01, and AUC

    This works regardless of whether val classes match training classes.
    """
    model.eval()

    # Extract all embeddings
    all_embeddings = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)

        with autocast(device_type='cuda', enabled=config.use_amp):
            # Use backbone only (no ArcFace head) for embeddings
            if hasattr(model, 'backbone'):
                emb = model.backbone(images)
            else:
                emb = model(images)

        # L2 normalize embeddings
        emb = F.normalize(emb, p=2, dim=1)
        all_embeddings.append(emb.cpu())
        all_labels.append(labels.cpu())

    all_embeddings = torch.cat(all_embeddings)
    all_labels = torch.cat(all_labels)
    print(f"   📊 Extracted {len(all_embeddings):,} embeddings ({all_embeddings.shape[1]}-D)")

    # Build per-class index for pair generation
    from collections import defaultdict
    class_indices = defaultdict(list)
    for idx, label in enumerate(all_labels.tolist()):
        class_indices[label].append(idx)

    # Only use classes with at least 2 samples (needed for positive pairs)
    valid_classes = [c for c, idxs in class_indices.items() if len(idxs) >= 2]
    all_class_list = list(class_indices.keys())

    if len(valid_classes) < 2:
        print("   ⚠️ Not enough classes with 2+ samples for pair verification")
        return {
            'verification_accuracy': 0, 'auc': 0, 'best_threshold': 0.5,
            'tar_at_far001': 0, 'pos_sim_mean': 0, 'pos_sim_std': 0,
            'neg_sim_mean': 0, 'neg_sim_std': 0,
        }

    import random
    random.seed(42 + len(all_embeddings))  # Varying seed per call

    positives_sims = []
    negatives_sims = []
    half_pairs = num_pairs // 2

    # Generate positive pairs (same identity)
    for _ in range(half_pairs):
        cls = random.choice(valid_classes)
        i, j = random.sample(class_indices[cls], 2)
        sim = F.cosine_similarity(all_embeddings[i].unsqueeze(0),
                                   all_embeddings[j].unsqueeze(0)).item()
        positives_sims.append(sim)

    # Generate negative pairs (different identities)
    for _ in range(half_pairs):
        cls1, cls2 = random.sample(all_class_list, 2)
        i = random.choice(class_indices[cls1])
        j = random.choice(class_indices[cls2])
        sim = F.cosine_similarity(all_embeddings[i].unsqueeze(0),
                                   all_embeddings[j].unsqueeze(0)).item()
        negatives_sims.append(sim)

    # Compute metrics
    all_sims = positives_sims + negatives_sims
    all_true = [1] * len(positives_sims) + [0] * len(negatives_sims)

    # Find optimal threshold (maximize accuracy)
    best_acc = 0
    best_threshold = 0.5
    for threshold in np.arange(-0.2, 1.01, 0.01):
        preds = [1 if s >= threshold else 0 for s in all_sims]
        acc = sum(p == t for p, t in zip(preds, all_true)) / len(all_true)
        if acc > best_acc:
            best_acc = acc
            best_threshold = threshold

    # TAR @ FAR = 0.01 (True Accept Rate at 1% False Accept Rate)
    neg_sorted = sorted(negatives_sims, reverse=True)
    far_idx = max(1, int(0.01 * len(neg_sorted))) - 1
    threshold_far001 = neg_sorted[far_idx] if neg_sorted else 0.5
    tar_at_far001 = sum(1 for s in positives_sims if s >= threshold_far001) / max(1, len(positives_sims))

    # AUC
    try:
        from sklearn.metrics import roc_auc_score
        auc = roc_auc_score(all_true, all_sims)
    except Exception:
        auc = best_acc  # Fallback

    # Distribution stats
    pos_mean = np.mean(positives_sims) if positives_sims else 0
    pos_std = np.std(positives_sims) if positives_sims else 0
    neg_mean = np.mean(negatives_sims) if negatives_sims else 0
    neg_std = np.std(negatives_sims) if negatives_sims else 0

    print(f"   📊 Pos sim: {pos_mean:.3f} ± {pos_std:.3f} | Neg sim: {neg_mean:.3f} ± {neg_std:.3f}")

    return {
        'verification_accuracy': best_acc * 100,
        'auc': auc,
        'best_threshold': best_threshold,
        'tar_at_far001': tar_at_far001 * 100,
        'pos_sim_mean': pos_mean,
        'pos_sim_std': pos_std,
        'neg_sim_mean': neg_mean,
        'neg_sim_std': neg_std,
    }

print("✅ Embedding-based verification validation defined")
print("   Metrics: Pair accuracy, AUC, TAR@FAR=0.01, similarity distributions")

## 8. Resume from Checkpoint (Optional)

In [ ]:
# Try to resume from checkpoint
start_epoch = 0
global_step = 0

latest_ckpt = find_latest_checkpoint(CHECKPOINT_DIR)
if latest_ckpt:
    start_epoch, global_step = load_checkpoint(latest_ckpt, model, optimizer, scaler)
else:
    print("📭 No checkpoint found, starting fresh")

## 9. 🔥 MAIN TRAINING LOOP - Run This to Start Training!

In [ ]:
# ============================================================================
# 🔄 TRAIN EPOCH FUNCTION (Fixed for PyTorch 2.x + ArcFace accuracy fix)
# ============================================================================

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
import math

def train_epoch(model, train_loader, optimizer, scaler, epoch, global_step, config, warmup_steps=0, total_steps=0):
    """
    Train for one epoch with mixed precision, gradient clipping, and warmup.
    Accuracy is measured on raw cosine logits (before ArcFace margin penalty).
    """
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.num_epochs}")

    for batch_idx, (images, labels) in enumerate(pbar):
        images = images.to(device)
        labels = labels.to(device)

        # Learning rate warmup
        if warmup_steps > 0 and global_step < warmup_steps:
            lr_scale = min(1.0, (global_step + 1) / warmup_steps)
            for pg in optimizer.param_groups:
                pg['lr'] = config.base_lr * lr_scale
        elif total_steps > 0 and global_step >= warmup_steps:
            # Cosine annealing after warmup
            progress = (global_step - warmup_steps) / max(1, total_steps - warmup_steps)
            lr = config.min_lr + 0.5 * (config.base_lr - config.min_lr) * (1 + math.cos(math.pi * progress))
            for pg in optimizer.param_groups:
                pg['lr'] = lr

        optimizer.zero_grad()

        # Forward with AMP
        with torch.amp.autocast('cuda', enabled=config.use_amp):
            logits = model(images, labels)
            loss = F.cross_entropy(logits, labels)

        # Backward pass with gradient scaling
        scaler.scale(loss).backward()

        # Gradient clipping
        if config.grad_clip > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)

        scaler.step(optimizer)
        scaler.update()

        # ✅ FIX: Compute accuracy on RAW cosine logits (before ArcFace margin)
        # ArcFace margin penalizes the correct class → argmax on margin logits = always wrong
        # Raw cosine accuracy shows the model's TRUE discriminative ability
        with torch.no_grad():
            if hasattr(model, 'head') and model.head._last_cos is not None:
                _, predicted = model.head._last_cos.max(1)
            else:
                _, predicted = logits.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        total_loss += loss.item()
        global_step += 1

        # Update progress bar
        avg_loss = total_loss / (batch_idx + 1)
        acc = 100.0 * correct / total
        current_lr = optimizer.param_groups[0]['lr']
        pbar.set_postfix({
            'loss': f'{avg_loss:.4f}',
            'acc': f'{acc:.2f}%',
            'lr': f'{current_lr:.2e}',
            'step': global_step
        })

        # Periodic checkpoint
        if config.checkpoint_every_n_steps > 0 and global_step % config.checkpoint_every_n_steps == 0:
            if 'save_checkpoint' in globals():
                try:
                    save_checkpoint(
                        CHECKPOINT_DIR / f'checkpoint_step_{global_step}.pth',
                        model, optimizer, scaler, epoch, global_step, avg_loss,
                        verbose=False
                    )
                    print(f"\n   💾 Checkpoint saved at step {global_step}")
                except:
                    pass

    # Return epoch metrics
    avg_loss = total_loss / len(train_loader)
    accuracy = 100.0 * correct / total

    return avg_loss, accuracy, global_step

print("✅ train_epoch function defined (raw cosine accuracy + PyTorch 2.x)")

In [ ]:
# ============================================================================
# 🔥 MAIN TRAINING LOOP (with ArcFace margin warmup + robust model saving)
# ============================================================================

print("\n" + "="*70)
print("🔥 STARTING TRAINING")
print("="*70)
print(f"Start epoch: {start_epoch}, Start step: {global_step}")
print(f"Total epochs: {config.num_epochs}")
print(f"Checkpoints → {CHECKPOINT_DIR}")

# ✅ Margin warmup config
margin_start = getattr(config, 'arcface_margin_start', 0.0)
margin_end = getattr(config, 'arcface_margin_end', config.arcface_margin)
margin_warmup_epochs = getattr(config, 'margin_warmup_epochs', 5)
print(f"ArcFace margin warmup: {margin_start} → {margin_end} over {margin_warmup_epochs} epochs\n")

best_val_acc = 0.0
best_train_acc = 0.0
best_embed_acc = 0.0
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'embed_acc': [],
}

try:
    for epoch in range(start_epoch, config.num_epochs):
        epoch_start = time.time()

        # =================================================================
        # ✅ MARGIN WARMUP: gradually increase ArcFace margin
        # =================================================================
        if epoch < margin_warmup_epochs:
            current_margin = margin_start + (margin_end - margin_start) * (epoch / max(1, margin_warmup_epochs))
        else:
            current_margin = margin_end

        if hasattr(model, 'head') and hasattr(model.head, 'set_margin'):
            model.head.set_margin(current_margin)
            print(f"   📐 ArcFace margin: {current_margin:.3f}")
        elif hasattr(model, 'head'):
            model.head.m = current_margin
            print(f"   📐 ArcFace margin: {current_margin:.3f}")

        # =================================================================
        # TRAINING
        # =================================================================
        train_loss, train_acc, global_step = train_epoch(
            model, train_loader, optimizer, scaler,
            epoch, global_step, config, warmup_steps, total_steps
        )

        # =================================================================
        # VALIDATION (classification-based — requires matching class indices)
        # =================================================================
        has_val = 'val_loader' in globals() and val_loader is not None
        if has_val:
            try:
                if len(val_dataset) > 0:
                    val_loss, val_acc = validate(model, val_loader, config)
                else:
                    val_loss, val_acc = 0, 0
                    has_val = False
            except Exception as e:
                print(f"   ⚠️ Validation error: {e}")
                val_loss, val_acc = 0, 0
                has_val = False
        else:
            val_loss, val_acc = 0, 0
            print("   ⏭️ Skipping validation (no val_loader)")

        # =================================================================
        # EMBEDDING VALIDATION (every 2 epochs — the REAL face rec metric)
        # =================================================================
        embed_acc = 0.0
        if has_val and (epoch + 1) % 2 == 0:
            try:
                embed_metrics = validate_embeddings(model, val_loader, config)
                embed_acc = embed_metrics['verification_accuracy']
                print(f"   🎯 Embed Verification Acc: {embed_acc:.2f}%")
                print(f"   🎯 TAR@FAR=1%: {embed_metrics['tar_at_far001']:.2f}%")
                print(f"   🎯 AUC: {embed_metrics['auc']:.4f}")
            except Exception as e:
                print(f"   ⚠️ Embedding validation error: {e}")
                embed_acc = 0.0

        epoch_time = time.time() - epoch_start

        # =================================================================
        # LOGGING
        # =================================================================
        print(f"\n📊 Epoch {epoch+1}/{config.num_epochs} | ⏱️ {epoch_time/60:.1f} min | margin={current_margin:.3f}")
        print(f"   📈 Train → Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
        if has_val:
            print(f"   📉 Val   → Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")
        if embed_acc > 0:
            print(f"   🎯 Embed → Verification Acc: {embed_acc:.2f}%")
        print(f"   📍 Global step: {global_step:,}")

        # Track history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['embed_acc'].append(embed_acc)

        # =================================================================
        # CHECKPOINTING
        # =================================================================
        save_checkpoint(
            CHECKPOINT_DIR / f'checkpoint_epoch_{epoch+1}.pth',
            model, optimizer, scaler, epoch+1, global_step, train_loss,
            verbose=True
        )

        save_checkpoint(
            CHECKPOINT_DIR / 'checkpoint_latest.pth',
            model, optimizer, scaler, epoch+1, global_step, train_loss,
            verbose=False
        )

        # ✅ Save best model — priority: embedding acc > val acc > train acc
        improved = False

        if embed_acc > best_embed_acc and embed_acc > 0:
            best_embed_acc = embed_acc
            improved = True
            print(f"   ⭐ NEW BEST! Embedding Verification Acc: {best_embed_acc:.2f}%")

        if has_val and val_acc > best_val_acc:
            best_val_acc = val_acc
            if not improved:
                improved = True
                print(f"   ⭐ NEW BEST! Val Acc: {best_val_acc:.2f}%")

        if train_acc > best_train_acc:
            best_train_acc = train_acc
            if not improved and (not has_val or best_val_acc < 1.0) and best_embed_acc < 1.0:
                improved = True
                print(f"   ⭐ NEW BEST! Train Acc: {best_train_acc:.2f}%")

        if improved:
            if hasattr(model, 'backbone'):
                torch.save(model.backbone.state_dict(), CHECKPOINT_DIR / 'best_backbone.pth')
                print(f"   💾 Saved best_backbone.pth")
            else:
                torch.save(model.state_dict(), CHECKPOINT_DIR / 'best_model.pth')
                print(f"   💾 Saved best_model.pth")

    # =========================================================================
    # TRAINING COMPLETE — Final Evaluation
    # =========================================================================
    print("\n" + "="*70)
    print("✅ TRAINING COMPLETE!")
    print("="*70)
    print(f"Best train accuracy:      {best_train_acc:.2f}%")
    if best_val_acc > 0:
        print(f"Best val accuracy:        {best_val_acc:.2f}%")
    if best_embed_acc > 0:
        print(f"Best embedding ver. acc:  {best_embed_acc:.2f}%")
    print(f"Total steps: {global_step:,}")
    print(f"Checkpoints saved to: {CHECKPOINT_DIR}")

    # Final comprehensive embedding evaluation
    if has_val:
        print("\n🔬 Final Embedding Evaluation (5000 pairs)...")
        try:
            final_metrics = validate_embeddings(model, val_loader, config, num_pairs=5000)
            print(f"   Verification Acc: {final_metrics['verification_accuracy']:.2f}% @ threshold={final_metrics['best_threshold']:.3f}")
            print(f"   TAR@FAR=1%: {final_metrics['tar_at_far001']:.2f}%")
            print(f"   AUC: {final_metrics['auc']:.4f}")
            print(f"   Pos similarity: {final_metrics['pos_sim_mean']:.3f} ± {final_metrics['pos_sim_std']:.3f}")
            print(f"   Neg similarity: {final_metrics['neg_sim_mean']:.3f} ± {final_metrics['neg_sim_std']:.3f}")
        except Exception as e:
            print(f"   ⚠️ Final evaluation error: {e}")

except KeyboardInterrupt:
    print(f"\n\n⚠️ Training interrupted at epoch {epoch+1}/{config.num_epochs}")
    print(f"Progress saved at step {global_step:,}")
    print(f"Resume by re-running this cell")

except Exception as e:
    print(f"\n❌ Training error:")
    print(f"{type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

## 10. Training Curves Visualization

In [ ]:
import matplotlib.pyplot as plt

if history['train_loss']:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    epochs = list(range(1, len(history['train_loss']) + 1))

    # Loss plot
    axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train', linewidth=2, markersize=4)
    if any(history['val_loss']):
        axes[0].plot(epochs, history['val_loss'], 'r-s', label='Val', linewidth=2, markersize=4)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)

    # Accuracy plot
    axes[1].plot(epochs, history['train_acc'], 'b-o', label='Train', linewidth=2, markersize=4)
    if any(history['val_acc']):
        axes[1].plot(epochs, history['val_acc'], 'r-s', label='Val', linewidth=2, markersize=4)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy (%)', fontsize=12)
    axes[1].set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()

    # Save the figure
    save_path = CHECKPOINT_DIR / 'training_curves.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"📊 Training curves saved to: {save_path}")

    plt.show()
else:
    print("⚠️ No training history available yet. Run training first!")

## 11. Training Complete - Next Steps

In [ ]:
print("="*70)
print("✅ MODEL TRAINING SECTION COMPLETE!")
print("="*70)

# ✅ FIX: Report both training and validation accuracy
_best_train = best_train_acc if 'best_train_acc' in dir() else max(history.get('train_acc', [0]))
_best_val = best_val_acc if 'best_val_acc' in dir() else max(history.get('val_acc', [0]))
_final_train_acc = history['train_acc'][-1] if history.get('train_acc') else 0
_final_val_acc = history['val_acc'][-1] if history.get('val_acc') else 0

print(f"""
📁 Files Saved to: {CHECKPOINT_DIR}

✨ Checkpoint Files:
   • best_backbone.pth (or best_model.pth) - Best model
   • checkpoint_epoch_*.pth - Epoch checkpoints for resuming
   • checkpoint_latest.pth - Latest training state
   • training_curves.png - Loss & accuracy plots

📊 Training Results:
   • Best Training Accuracy:    {_best_train:.2f}%
   • Best Validation Accuracy:  {_best_val:.2f}%
   • Final Training Accuracy:   {_final_train_acc:.2f}%
   • Final Validation Accuracy: {_final_val_acc:.2f}%
   • Total Training Steps: {global_step:,}
   • Total Epochs Completed: {len(history['train_loss'])}

🚀 Next Steps:
   1. Proceed to Model Evaluation section below
   2. Benchmark on LFW, AgeDB, CFP-FP datasets
   3. Export model to ONNX for deployment
   4. Integrate with attendance system

💡 To Resume Training:
   • Simply re-run the training loop cell
   • It will automatically load from checkpoint_latest.pth
   • Training will continue from where it stopped

""")
print("="*70)